## Query source LH, save to target LH + adjust to semantic model
This notebook queries the source lakehouse based on a defined SQL query. After that, it saves the result set to the target lakehouse. 

Next, it uses Semantic Link Labs to add the resulting table from the target lakehouse to the defined semantic model. 

In [1]:
%pip install semantic-link-labs
import sempy_labs as labs
import sempy.fabric as fabric

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 8, Finished, Available, Finished, False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.4/973.4 kB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 kB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.8/687.8 kB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.0/856.0 kB 119.5 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 1.30.2
    Not uninstalling azure-core 

In [2]:
# Define source and target lakehouses (schemas/databases)
source_lakehouse = "LH_STORE_Silver"
target_lakehouse = "LH_STORE_Gold"

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 10, Finished, Available, Finished, False)

#### Create table in destination lakehouse

In [3]:
# Define source table and target table names
source_table = "internetsales"
target_table = "sales"  # new table in Gold to save results

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 11, Finished, Available, Finished, False)

In [4]:
sql_query = f"""
SELECT 
    SalesOrderNumber AS OrderNumber,
    ProductKey, 
    OrderDateKey, 
    ShipDateKey,
    CustomerKey,
    OrderQuantity AS Quantity,
    SalesAmount AS Price
FROM {source_lakehouse}.{source_table}
"""

# Run the SQL query and get a DataFrame
df = spark.sql(sql_query)

# Show first 5 rows of the resulting DataFrame
df.show(5)


StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 12, Finished, Available, Finished, False)

+-----------+----------+------------+-----------+-----------+--------+--------------------+
|OrderNumber|ProductKey|OrderDateKey|ShipDateKey|CustomerKey|Quantity|               Price|
+-----------+----------+------------+-----------+-----------+--------+--------------------+
|    SO51555|       480|    20190623|   20190630|      11037|       1|2.290000000000000000|
|    SO58176|       480|    20191018|   20191025|      14338|       1|2.290000000000000000|
|    SO64622|       480|    20200121|   20200128|      11951|       1|2.290000000000000000|
|    SO66652|       480|    20200220|   20200227|      16329|       1|2.290000000000000000|
|    SO69217|       480|    20200328|   20200404|      16147|       1|2.290000000000000000|
+-----------+----------+------------+-----------+-----------+--------+--------------------+
only showing top 5 rows



In [5]:
# Write the DataFrame as a new table in the target lakehouse
df.write.mode("overwrite").saveAsTable(f"{target_lakehouse}.{target_table}")

print(f"✅ Saved filtered data to table '{target_table}' in lakehouse '{target_lakehouse}'")

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 13, Finished, Available, Finished, False)

✅ Saved filtered data to table 'sales' in lakehouse 'LH_STORE_Gold'


#### Update Semantic Model

In [6]:
semanticmodel_name = "Live Demo Data Grillen"

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 14, Finished, Available, Finished, False)

In [7]:
labs.directlake.add_table_to_direct_lake_semantic_model(
    dataset= semanticmodel_name,
    table_name= target_table, # reusing the target table name, as the gold lakehouse should already be self explanatory
    lakehouse_table_name= target_table,
    refresh= False,
    workspace= None, # if not specified, will be the same workspace as the notebook runs
    columns= None
)

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 15, Finished, Available, Finished, False)

🟢 The 'sales' table has been added to the 'Live Demo Data Grillen' semantic model within the 'Reaching maximum automation' workspace.


ArgumentException: An object with name 'DatabaseQuery' does not exist in the collection. (Parameter 'name')
   at Microsoft.AnalysisServices.Tabular.NamedMetadataObjectCollection`2.get_Item(String name)

In [12]:
# Refresh semantic model when all is done
fabric.refresh_dataset(
    workspace= None,
    dataset= semanticmodel_name, 
    refresh_type= 'automatic'
)

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 21, Finished, Available, Finished, False)

'3d590c3d-2021-41be-96f2-175e70f200a3'

## Add relationships to the model
After we added the tables, we need to create relationships to the dimension tables

In [24]:
# setup connection to Tabular Object Model (TOM) and list tables as example
with labs.tom.connect_semantic_model(dataset=semanticmodel_name, readonly=False, workspace=None) as tom:
    for t in tom.model.Tables:
        print(t.Name)

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 35, Finished, Available, Finished, False)

_measures
sales
date
product


In [25]:
# Define relationships as they should be added to the model
# From should always be the many side of the relationship / fact table (current limitation)

relationships = [
    {'FromTable' : 'sales', 'FromColumn':'ProductKey', 'ToTable':'product',  'ToColumn':'ProductKey'},
    {'FromTable' : 'sales', 'FromColumn':'OrderDateKey', 'ToTable':'date',  'ToColumn':'DateKey'}
]

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 36, Finished, Available, Finished, False)

In [26]:
# List current existing relationships from the semantic model
currentrelationships = fabric.list_relationships(semanticmodel_name)

# Add a new column to store results
currentrelationships["relationship_name"] = ""

# add relationship concatenation in a new column to the dataframe
for idx, row in currentrelationships.iterrows():
        relationship_name = labs.create_relationship_name(
            from_table=row["From Table"],
            from_column=row["From Column"],
            to_table=row["To Table"],
            to_column=row["To Column"]
        )
        currentrelationships.at[idx, "relationship_name"] = relationship_name

# Show the updated DataFrame
display(currentrelationships)

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52017dbf-99c4-4b80-bfd8-ee4dd3ac666a)

In [27]:
# Validating if relationship already exists in the model

for relationship in relationships:
    relationship_name = labs.create_relationship_name(
            from_table=relationship["FromTable"],
            from_column=relationship["FromColumn"],
            to_table=relationship["ToTable"],
            to_column=relationship["ToColumn"]
        )
    # returning results of validation
    does_relationship_exist = relationship_name in currentrelationships['relationship_name'].tolist()
    print(f'Relationship: {relationship_name}; Does it exist? {does_relationship_exist}')
    
    # if relationship does not exist, create relationship
    if not does_relationship_exist:
        with labs.tom.connect_semantic_model(dataset=semanticmodel_name, readonly=False, workspace=None) as tom:
            tom.add_relationship(
                from_table= relationship["FromTable"], 
                from_column= relationship["FromColumn"], 
                to_table= relationship["ToTable"], 
                to_column= relationship["ToColumn"],
                from_cardinality= 'Many', # ‘Many’, ‘One’, ‘None’
                to_cardinality = 'One', # ‘Many’, ‘One’, ‘None’
                cross_filtering_behavior= 'OneDirection', # ‘OneDirection’, ‘BothDirections’
                is_active= True, 
                security_filtering_behavior= None, 
                rely_on_referential_integrity= False)
            print(f'✅ Relationship {relationship_name} has been created') 
 
 # Rerunning this cell may result in errors, as the cell above (listing the current relationships) forms an input. This input may be cached and therefore return errors. 
 # If the relationship already exists, the error lists something along the lines of "ambiguous path between .... "

StatementMeta(, d8cca75b-9ad4-411a-a0af-e98ddf0063a3, 38, Finished, Available, Finished, False)

Relationship: 'sales'[ProductKey] -> 'product'[ProductKey]; Does it exist? False
✅ Relationship 'sales'[ProductKey] -> 'product'[ProductKey] has been created
Relationship: 'sales'[OrderDateKey] -> 'date'[DateKey]; Does it exist? False
✅ Relationship 'sales'[OrderDateKey] -> 'date'[DateKey] has been created
